# TP Qualité des données IRVE

## Objectif
Auditer, nettoyer et contrôler la qualité du fichier consolidé des bornes de recharge pour véhicules électriques.

## Étapes
1. Chargement et exploration
2. Audit qualité initial
3. Étude du grain et des doublons
4. Contrôle des coordonnées GPS
5. Nettoyage et enrichissement
6. Validation automatisée
7. Monitoring

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

print("Imports réussis")

Imports réussis


# Partie 1 — Profiling et audit initial

## Objectif

Cette première partie consiste à observer et mesurer la qualité du dataset IRVE avant toute transformation.

Les analyses réalisées portent sur :

* les dimensions du dataset ;
* les noms et les types des colonnes ;
* les valeurs manquantes ;
* les doublons stricts et les identifiants répétés ;
* les coordonnées géographiques douteuses ;
* les puissances aberrantes ;
* les dates absentes, invalides ou futures ;
* les six dimensions de la qualité des données ;
* le calcul d’un score qualité initial.

À ce stade, aucune donnée n’est supprimée ou modifiée. Le score obtenu servira de référence pour comparer la qualité avant et après nettoyag


In [2]:
FILE_PATH = "consolidation-etalab-schema-irve-statique-v-2.3.1-20260625.csv"

df = pd.read_csv(
    FILE_PATH,
    low_memory=False
)

print("Dimensions :", df.shape)

Dimensions : (227232, 52)


## Description des colonnes du dataset IRVE

Le dataset contient **52 colonnes** organisées autour de plusieurs niveaux d’information : les acteurs, les stations, les points de charge, les caractéristiques techniques, les modalités d’accès, la traçabilité et les données consolidées.

### 1. Informations sur les acteurs

* `nom_amenageur` : nom du responsable ou propriétaire de l’installation.
* `siren_amenageur` : numéro SIREN de l’aménageur.
* `contact_amenageur` : contact de l’aménageur.
* `nom_operateur` : nom de l’opérateur qui exploite la borne.
* `contact_operateur` : contact de l’opérateur.
* `telephone_operateur` : numéro de téléphone de l’opérateur.
* `nom_enseigne` : nom commercial visible par l’utilisateur.

### 2. Informations sur la station

Une station peut contenir plusieurs points de charge.

* `id_station_itinerance` : identifiant interopérable de la station.
* `id_station_local` : identifiant interne de la station.
* `nom_station` : nom de la station.
* `implantation_station` : type d’implantation de la station.
* `adresse_station` : adresse postale.
* `code_insee_commune` : code INSEE de la commune.
* `coordonneesXY` : coordonnées géographiques brutes.
* `nbre_pdc` : nombre de points de charge déclarés dans la station.

### 3. Informations sur le point de charge

* `id_pdc_itinerance` : identifiant interopérable du point de charge.
* `id_pdc_local` : identifiant local du point de charge.
* `puissance_nominale` : puissance maximale du point de charge, normalement exprimée en kW.

### 4. Types de prises

Ces colonnes indiquent si le point de charge propose un type de connecteur donné.

* `prise_type_ef` : prise domestique de type E/F.
* `prise_type_2` : connecteur de type 2.
* `prise_type_combo_ccs` : connecteur Combo CCS.
* `prise_type_chademo` : connecteur CHAdeMO.
* `prise_type_autre` : autre type de prise.
* `cable_t2_attache` : présence d’un câble de type 2 attaché.

### 5. Accès, paiement et utilisation

* `gratuit` : recharge gratuite ou payante.
* `paiement_acte` : paiement ponctuel possible.
* `paiement_cb` : paiement par carte bancaire possible.
* `paiement_autre` : autre moyen de paiement.
* `tarification` : description de la tarification.
* `condition_acces` : conditions d’accès à la station.
* `reservation` : possibilité de réserver.
* `horaires` : horaires d’ouverture.
* `accessibilite_pmr` : accessibilité aux personnes à mobilité réduite.
* `restriction_gabarit` : restrictions de hauteur, longueur ou poids.
* `station_deux_roues` : station adaptée aux deux-roues.

### 6. Raccordement électrique

* `raccordement` : type de raccordement au réseau électrique.
* `num_pdl` : numéro du point de livraison électrique.

### 7. Dates et observations

* `date_mise_en_service` : date de mise en service du point de charge.
* `observations` : remarques complémentaires.
* `date_maj` : date de mise à jour déclarée.
* `last_modified` : date de dernière modification de la ressource.
* `created_at` : date de création de la ligne dans le système de consolidation.

### 8. Traçabilité Data.gouv

* `datagouv_dataset_id` : identifiant du jeu de données source.
* `datagouv_resource_id` : identifiant de la ressource source.
* `datagouv_organization_or_owner` : organisation ayant publié les données.

### 9. Colonnes consolidées

* `consolidated_longitude` : longitude consolidée.
* `consolidated_latitude` : latitude consolidée.
* `consolidated_code_postal` : code postal consolidé.
* `consolidated_commune` : commune consolidée.
* `consolidated_is_lon_lat_correct` : indicateur de validité des coordonnées.
* `consolidated_is_code_insee_verified` : indique si le code INSEE a été vérifié.
* `consolidated_is_code_insee_modified` : indique si le code INSEE a été corrigé.




## 2. Qualité des données

Cette section permet d'identifier les principaux problèmes de qualité avant tout nettoyage :

- les valeurs manquantes ;
- les types de données ;
- les coordonnées géographiques douteuses ;
- les puissances aberrantes ;
- les dates absentes, invalides ou futures.


In [3]:
# Calcul du nombre et du pourcentage de valeurs manquantes par colonne

audit_manquants = pd.DataFrame({
    "nombre_manquants": df.isna().sum(),
    "pourcentage_manquants": (df.isna().mean() * 100).round(2)
})

audit_manquants = audit_manquants.sort_values(
    by="pourcentage_manquants",
    ascending=False
)

audit_manquants.head(52)

,nombre_manquants,pourcentage_manquants
observations,180712,79.53
tarification,170887,75.20
num_pdl,111260,48.96
cable_t2_attache,106741,46.97
consolidated_code_postal,92870,40.87
date_mise_en_service,92271,40.61
siren_amenageur,90985,40.04
raccordement,79307,34.90
id_pdc_local,78678,34.62
consolidated_commune,76393,33.62


Comme le tableau est trié du plus grand au plus petit pourcentage de valeurs manquantes, les 20 premières lignes montrent directement les colonnes les plus problématiques.

In [4]:
audit_manquants.head(20)

,nombre_manquants,pourcentage_manquants
observations,180712,79.53
tarification,170887,75.20
num_pdl,111260,48.96
cable_t2_attache,106741,46.97
consolidated_code_postal,92870,40.87
date_mise_en_service,92271,40.61
siren_amenageur,90985,40.04
raccordement,79307,34.90
id_pdc_local,78678,34.62
consolidated_commune,76393,33.62


### 2.2 Analys des types de données

Cette étape permet de vérifier le type de chaque colonne et le nombre de valeurs distinctes.

L'objectif est d'identifier notamment :

- les dates stockées comme texte ;
- les codes postaux et les identifiants stockés comme nombres ;
- les colonnes booléennes contenant plusieurs formats ;
- les variables catégorielles présentant un nombre limité de modalités.


In [5]:
# Types des colonnes
types_colonnes = pd.DataFrame({
    "type": df.dtypes,
    "nombre_valeurs_uniques": df.nunique(dropna=False)
})

display(types_colonnes)

,type,nombre_valeurs_uniques
nom_amenageur,object,4409
siren_amenageur,float64,2941
contact_amenageur,object,909
nom_operateur,object,359
contact_operateur,object,356
telephone_operateur,object,672
nom_enseigne,object,6985
id_station_itinerance,object,63946
id_station_local,object,55464
nom_station,object,45977


### Interprétation des types

L'analyse met en évidence plusieurs points de vigilance :

- les colonnes `date_mise_en_service`, `date_maj`, `last_modified` et `created_at` sont stockées comme texte ;
- `consolidated_code_postal` est stocké comme nombre décimal alors qu'un code postal doit être traité comme du texte ;
- `siren_amenageur` est également stocké comme nombre alors qu'il s'agit d'un identifiant ;
- plusieurs colonnes attendues comme booléennes présentent plusieurs formes de valeurs ;
- les indicateurs `consolidated_is_*` sont correctement reconnus comme booléens.

Ces constats concernent principalement les dimensions de validité et de cohérence.


In [6]:
# Contrôle des coordonnées signalées comme correctes ou incorrectes

controle_coordonnees = (
    df["consolidated_is_lon_lat_correct"]
    .value_counts(dropna=False)
)

display(controle_coordonnees)

consolidated_is_lon_lat_correct
True     145508
False     81724
Name: count, dtype: int64

### Résultat du contrôle des coordonnées

Sur les 227 232 lignes du dataset :

- 145 508 lignes ont des coordonnées considérées comme correctes ;
- 81 724 lignes ont des coordonnées signalées comme incorrectes.

Cela représente environ 35,96 % du dataset.

Ce résultat concerne principalement la dimension d'exactitude.

In [7]:
puissance = pd.to_numeric(
    df["puissance_nominale"],
    errors="coerce"
)

mask_puissance_aberrante = (
    (puissance <= 0)
    | (puissance > 1000)
)

print("Puissances manquantes :", puissance.isna().sum())
print("Puissances inférieures ou égales à 0 :", (puissance <= 0).sum())
print("Puissances supérieures à 1000 kW :", (puissance > 1000).sum())
print("Total des puissances aberrantes :", mask_puissance_aberrante.sum())

Puissances manquantes : 0
Puissances inférieures ou égales à 0 : 5282
Puissances supérieures à 1000 kW : 975
Total des puissances aberrantes : 6257


### Résultat du contrôle des puissances

La colonne `puissance_nominale` ne contient aucune valeur manquante.

6 257 valeurs sont considérées comme aberrantes selon la règle utilisée :

- 5 282 valeurs sont inférieures ou égales à 0 kW ;
- 975 valeurs sont supérieures à 1 000 kW.

Ces anomalies concernent principalement la dimension de validité.

Piste probable : puissances très élevées peuvent correspondre à une erreur d'unité (W vs kW)

In [8]:
# Contrôle des colonnes de dates

colonnes_dates = [
    "date_mise_en_service",
    "date_maj",
    "last_modified",
    "created_at"
]

for colonne in colonnes_dates:
    dates = pd.to_datetime(
        df[colonne],
        errors="coerce",
        utc=True
    )

    print(f"\n--- {colonne} ---")
    print("Valeurs absentes ou invalides :", dates.isna().sum())
    print("Date minimale :", dates.min())
    print("Date maximale :", dates.max())


--- date_mise_en_service ---
Valeurs absentes ou invalides : 92271
Date minimale : 1900-01-01 00:00:00+00:00
Date maximale : 2026-06-24 00:00:00+00:00

--- date_maj ---
Valeurs absentes ou invalides : 292
Date minimale : 2012-09-10 00:00:00+00:00
Date maximale : 2026-12-30 00:00:00+00:00

--- last_modified ---
Valeurs absentes ou invalides : 33877
Date minimale : 2024-02-09 06:22:40.878000+00:00
Date maximale : 2026-06-24 21:00:21.100000+00:00

--- created_at ---
Valeurs absentes ou invalides : 0
Date minimale : 2021-05-10 21:01:29.234000+00:00
Date maximale : 2026-06-24 15:14:32.641000+00:00


### Résultat du contrôle des dates

L'analyse des colonnes de dates met en évidence plusieurs problèmes de qualité.

#### `date_mise_en_service`

- 92 271 valeurs sont absentes ou non convertibles ;
- la date minimale est le 1er janvier 1900 ;
- la date maximale est le 24 juin 2026.

La date de 1900 semble peu crédible pour une borne de recharge électrique. Elle peut correspondre à une valeur par défaut ou à une erreur de saisie.

#### `date_maj`

- 292 valeurs sont absentes ou invalides ;
- la date minimale est le 10 septembre 2012 ;
- la date maximale est le 30 décembre 2026.

La date maximale est future par rapport à la date du fichier. Elle doit donc être signalée comme anomalie de cohérence ou de fraîcheur.

#### `last_modified`

- 33 877 valeurs sont absentes ou invalides ;
- les dates observées vont du 9 février 2024 au 24 juin 2026.

Cette colonne décrit la dernière modification de la ressource source sur Data.gouv.

#### `created_at`

- aucune valeur n'est absente ;
- les dates vont du 10 mai 2021 au 24 juin 2026.

Cette colonne est la plus complète parmi les colonnes de dates analysées.

### Dimensions concernées

- Complétude : dates absentes ;
- Validité : dates non convertibles ou peu plausibles ;
- Cohérence : dates futures ou dates par défaut ;
- Fraîcheur : dates de mise à jour anciennes.

In [9]:
aujourdhui = pd.Timestamp.now(tz="UTC")

for colonne in colonnes_dates:
    dates = pd.to_datetime(
        df[colonne],
        errors="coerce",
        utc=True
    )

    print(
        colonne,
        "- dates futures :",
        (dates > aujourdhui).sum()
    )

date_mise_en_service - dates futures : 0
date_maj - dates futures : 56
last_modified - dates futures : 0
created_at - dates futures : 0


### Dates futures

La colonne `date_maj` contient 56 dates postérieures à la date actuelle.

Ces valeurs sont considérées comme des anomalies de cohérence temporelle et seront vérifiées lors du nettoyage.

In [10]:
#1 grille_audit
grille_audit = pd.DataFrame([
    {
        "Dimension": "Complétude",
        "Problème": "Valeurs manquantes",
        "Colonne": "Plusieurs colonnes",
        "Résultat": "Présence importante de valeurs manquantes",
        "Gravité": "Élevée"
    },
    {
        "Dimension": "Unicité",
        "Problème": "Identifiants répétés",
        "Colonne": "id_pdc_itinerance",
        "Résultat": "67 094 occurrences supplémentaires",
        "Gravité": "Élevée"
    },
    {
        "Dimension": "Exactitude",
        "Problème": "Coordonnées douteuses",
        "Colonne": "consolidated_is_lon_lat_correct",
        "Résultat": "81 724 lignes, soit 35,96 %",
        "Gravité": "Élevée"
    },
    {
        "Dimension": "Validité",
        "Problème": "Puissances aberrantes",
        "Colonne": "puissance_nominale",
        "Résultat": "6 257 valeurs",
        "Gravité": "Élevée"
    },
    {
        "Dimension": "Cohérence",
        "Problème": "Types et formats non homogènes",
        "Colonne": "Dates, booléens, adresses",
        "Résultat": "Dates en texte et problèmes d'encodage",
        "Gravité": "Moyenne"
    },
    {
        "Dimension": "Fraîcheur",
        "Problème": "Dates futures",
        "Colonne": "date_maj",
        "Résultat": "56 dates futures",
        "Gravité": "Moyenne"
    }
])

display(grille_audit)

,Dimension,Problème,Colonne,Résultat,Gravité
0,Complétude,Valeurs manquantes,Plusieurs colonnes,Présence importante de valeurs manquantes,Élevée
1,Unicité,Identifiants répétés,id_pdc_itinerance,67 094 occurrences supplémentaires,Élevée
2,Exactitude,Coordonnées douteuses,consolidated_is_lon_lat_correct,"81 724 lignes, soit 35,96 %",Élevée
3,Validité,Puissances aberrantes,puissance_nominale,6 257 valeurs,Élevée
4,Cohérence,Types et formats non homogènes,"Dates, booléens, adresses",Dates en texte et problèmes d'encodage,Moyenne
5,Fraîcheur,Dates futures,date_maj,56 dates futures,Moyenne


### Interprétation de la grille d'audit

L'audit met en évidence des problèmes sur les six dimensions de la qualité des données.

Les anomalies les plus importantes concernent :

- l'unicité, avec 67 094 occurrences supplémentaires de `id_pdc_itinerance` ;
- l'exactitude, avec 81 724 coordonnées signalées comme douteuses ;
- la validité, avec 6 257 puissances aberrantes ;
- la complétude, avec plusieurs colonnes fortement incomplètes.

Les problèmes de cohérence et de fraîcheur sont également présents, notamment à cause des formats non homogènes, des problèmes d'encodage et des dates futures.

## Score qualité initial

Le score initial est calculé sur les six dimensions de la qualité des données.

Chaque dimension est notée sur 100, puis une moyenne simple est calculée. Ce score sert de point de comparaison avec le score obtenu après nettoyage.

In [11]:
# 1. Complétude : pourcentage de cellules non manquantes
score_completude = df.notna().mean().mean() * 100


# 2. Unicité : proportion d'identifiants de points de charge uniques
score_unicite = (
    df["id_pdc_itinerance"].nunique(dropna=True)
    / len(df)
) * 100


# 3. Exactitude : proportion de coordonnées signalées comme correctes
score_exactitude = (
    df["consolidated_is_lon_lat_correct"].mean()
) * 100


# 4. Validité : proportion de puissances comprises entre 0 et 1000 kW
score_validite = (
    ~mask_puissance_aberrante
).mean() * 100


# 5. Fraîcheur : date_maj présente et non future
dates_maj = pd.to_datetime(
    df["date_maj"],
    errors="coerce",
    utc=True
)

aujourdhui = pd.Timestamp.now(tz="UTC")

score_fraicheur = (
    dates_maj.notna()
    & (dates_maj <= aujourdhui)
).mean() * 100

In [12]:
colonnes_booleennes = [
    "prise_type_ef",
    "prise_type_2",
    "prise_type_combo_ccs",
    "prise_type_chademo",
    "prise_type_autre",
    "gratuit",
    "paiement_acte",
    "paiement_cb",
    "reservation",
    "cable_t2_attache"
]

valeurs_booleennes_valides = {
    "true", "false",
    "vrai", "faux",
    "1", "0"
}

scores_coherence = []

for colonne in colonnes_booleennes:
    valeurs_normalisees = (
        df[colonne]
        .astype("string")
        .str.strip()
        .str.lower()
    )

    valeurs_coherentes = (
        valeurs_normalisees.isin(valeurs_booleennes_valides)
        | valeurs_normalisees.isna()
    )

    scores_coherence.append(valeurs_coherentes.mean() * 100)

score_coherence = sum(scores_coherence) / len(scores_coherence)

In [13]:
scores_initiaux = pd.Series({
    "Complétude": score_completude,
    "Validité": score_validite,
    "Cohérence (booléens)": score_coherence,
    "Unicité": score_unicite,
    "Exactitude": score_exactitude,
    "Fraîcheur": score_fraicheur
}).round(2)

score_qualite_initial = scores_initiaux.mean().round(2)

display(scores_initiaux.to_frame("Score sur 100"))

print(
    "Score qualité global initial :",
    score_qualite_initial,
    "/ 100"
)

,Score sur 100
Complétude,87.73
Validité,97.25
Cohérence (booléens),100.00
Unicité,70.47
Exactitude,64.03
Fraîcheur,99.85


Score qualité global initial : 86.56 / 100


## Justification des scores de qualité

### Complétude — 87,73/100

Ce score correspond au pourcentage moyen de cellules renseignées dans l’ensemble du dataset.

Il est relativement élevé, mais plusieurs colonnes restent fortement incomplètes, notamment :

* `observations` ;
* `tarification` ;
* `num_pdl` ;
* `consolidated_code_postal` ;
* `date_mise_en_service`.

Le score montre donc que la majorité des données sont présentes, mais que certaines informations importantes manquent encore.

### Validité — 97,25/100

Ce score mesure la proportion de puissances considérées comme valides selon la règle suivante :

* puissance strictement supérieure à 0 kW ;
* puissance inférieure ou égale à 1 000 kW.

Sur 227 232 lignes, 6 257 puissances ne respectent pas cette règle :

* 5 282 sont inférieures ou égales à 0 ;
* 975 sont supérieures à 1 000 kW.

Le score reste élevé, car la grande majorité des puissances se situe dans l’intervalle retenu.

Cette mesure doit toutefois être interprétée avec prudence, car certaines valeurs supérieures à 1 000 peuvent être exprimées en watts au lieu de kilowatts.

### Cohérence — 100/100

Ce score a été calculé uniquement à partir de la normalisation des colonnes supposées booléennes.

Après conversion en minuscules et suppression des espaces, les valeurs observées ont été considérées comme compatibles avec les formes autorisées :

* `true` ;
* `false` ;
* `vrai` ;
* `faux` ;
* `1` ;
* `0` ;
* valeurs manquantes.

Selon cette règle technique, toutes les valeurs contrôlées sont cohérentes, ce qui explique le score de 100/100.

Cependant, ce score est trop optimiste pour représenter toute la cohérence du dataset. D’autres anomalies ont été observées :

* caractères mal encodés dans les adresses ;
* différences de casse ;
* variantes d’écriture des communes et des stations ;
* informations contradictoires entre plusieurs sources ;
* puissances différentes pour un même identifiant.

Le score de cohérence mesure donc seulement un aspect limité de cette dimension. Il ne signifie pas que l’ensemble du dataset est parfaitement cohérent.

### Unicité — 70,47/100

Ce score correspond au rapport entre le nombre d’identifiants `id_pdc_itinerance` uniques et le nombre total de lignes.

Le dataset contient :

* 227 232 lignes ;
* 160 138 identifiants uniques ;
* 67 094 occurrences supplémentaires.

Le score est donc dégradé par le nombre important d’identifiants répétés.

Cependant, ces répétitions ne sont pas nécessairement des erreurs. Elles peuvent représenter plusieurs versions ou publications d’un même point de charge. Le score mesure ici la répétition des identifiants, mais pas encore la légitimité de chaque répétition.

### Exactitude — 64,03/100

Ce score correspond à la proportion de lignes pour lesquelles la colonne `consolidated_is_lon_lat_correct` vaut `True`.

Le dataset contient :

* 145 508 coordonnées considérées comme correctes ;
* 81 724 coordonnées signalées comme douteuses.

L’exactitude est donc la dimension la plus faible.

Ce score repose sur un indicateur déjà présent dans le fichier consolidé. Il signale un risque, mais ne permet pas à lui seul de savoir si chaque coordonnée est réellement fausse.

### Fraîcheur — 99,85/100

Ce score repose sur la colonne `date_maj`.

Une ligne est considérée comme fraîche lorsque :

* la date est présente ;
* elle peut être convertie en date ;
* elle n’est pas située dans le futur.

Le score est très élevé, car seules quelques lignes présentent une date manquante, invalide ou future.

Le dataset contient notamment 56 dates futures dans `date_maj`.

Cette mesure ne garantit toutefois pas que toutes les données sont réellement récentes. Une date valide mais ancienne peut toujours indiquer une information peu fraîche.

## Conclusion

Les scores permettent de résumer rapidement la qualité initiale du dataset, mais chacun dépend d’une règle de calcul précise.

Ils doivent donc être accompagnés d’une analyse détaillée, car un score élevé peut masquer certains problèmes non pris en compte par la règle utilisée.


## Conclusion du profiling et de l’audit initial

À ce stade, aucune transformation ni aucun nettoyage n’a été appliqué au dataset.

Le travail réalisé correspond uniquement à une phase d’audit, d’observation et de mesure. Nous avons :

* chargé le fichier CSV ;
* analysé les 52 colonnes ;
* compté les valeurs manquantes ;
* repéré les identifiants répétés ;
* contrôlé la qualité des coordonnées géographiques ;
* vérifié les puissances nominales ;
* examiné les colonnes de dates ;
* calculé plusieurs indicateurs de qualité.

Le score qualité global initial obtenu est de **86,56/100**.

Ce score représente une estimation de la qualité actuelle de la base avant nettoyage.
Il ne constitue pas une vérité absolue, car il dépend directement des règles et des seuils choisis pour chaque dimension :

* la complétude dépend du taux de cellules renseignées ;
* l’unicité dépend du nombre d’identifiants uniques ;
* l’exactitude dépend du contrôle des coordonnées ;
* la validité dépend des seuils retenus pour les puissances ;
* la fraîcheur dépend de la qualité de la colonne `date_maj` ;
* la cohérence dépend des règles appliquées aux formats et aux valeurs booléennes.

Le score de cohérence de 100/100 doit être interprété avec prudence. 
Des problèmes d’encodage, de casse, de formats et des différences entre sources ont été observés.
Cela montre qu’un score synthétique peut masquer certaines anomalies si les règles de calcul sont trop simples.

Ce score initial servira donc de référence pour mesurer l’amélioration de la qualité après les étapes de nettoyage et de transformation.


## Rapport automatique avec ydata-profiling

Un rapport automatique est généré afin d’obtenir une vue globale des types de données, des valeurs manquantes, des distributions, des doublons et des principales alertes qualité.

Le mode minimal est utilisé afin de limiter le temps de calcul sur ce dataset volumineux.

In [14]:
from ydata_profiling import ProfileReport

print("ydata-profiling importé avec succès")

ydata-profiling importé avec succès


C:\Users\Sara\AppData\Local\Temp\ipykernel_6080\2161875287.py:1: DeprecationWarning: 
    `import ydata_profiling` is deprecated and will not receive more updates. 
    Please install fg-data-profiling via `pip install fg-data-profiling` and use `import data_profiling` instead.
    
  from ydata_profiling import ProfileReport


In [ ]:
echantillon_profiling = df.sample(
    n=30000,
    random_state=42
)

profile = ProfileReport(
    echantillon_profiling,
    title="Profiling initial IRVE - échantillon de 30 000 lignes",
    minimal=True
)

profile.to_file("rapport_profiling_irve_initial.html")

print("Rapport généré : rapport_profiling_irve_initial.html")

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]


100%|██████████| 52/52 [00:03<00:00, 16.10it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

# Partie 2 — Étude du grain et des doublons

L'objectif est de comprendre pourquoi certains `id_pdc_itinerance` apparaissent plusieurs fois avant de définir une règle de dédoublonnage.

## 1. Compréhension du grain

L'objectif est de comprendre ce que représente une ligne avant de supprimer des données.

Une ligne semble représenter un point de charge provenant d'une ressource ou d'une livraison donnée.

Un même `id_pdc_itinerance` peut donc apparaître plusieurs fois sans que les lignes soient strictement identiques.


In [16]:
# Nombre de doublons stricts
doublons_stricts = df.duplicated().sum()

# Nombre d'identifiants uniques
nombre_ids_uniques = df["id_pdc_itinerance"].nunique()

# Nombre d'identifiants apparaissant plusieurs fois
comptage_ids = df["id_pdc_itinerance"].value_counts()

nombre_ids_repetes = (comptage_ids > 1).sum()

# Nombre de lignes appartenant à des groupes répétés
lignes_ids_repetes = comptage_ids[comptage_ids > 1].sum()

# Nombre de lignes en trop si on gardait une ligne par identifiant
lignes_potentiellement_en_trop = (
    comptage_ids[comptage_ids > 1] - 1
).sum()

print("Doublons stricts :", doublons_stricts)
print("Identifiants uniques :", nombre_ids_uniques)
print("Identifiants répétés :", nombre_ids_repetes)
print("Lignes appartenant à des groupes répétés :", lignes_ids_repetes)
print("Lignes potentiellement en trop :", lignes_potentiellement_en_trop)

Doublons stricts : 0
Identifiants uniques : 160138
Identifiants répétés : 63870
Lignes appartenant à des groupes répétés : 130964
Lignes potentiellement en trop : 67094


Un identifiant répété est une anomalie à analyser, mais ce n’est pas automatiquement une erreur. Il faut comparer les dates, les sources et les caractéristiques des lignes avant de décider.

In [17]:
# Afficher les 10 identifiants les plus répétés
comptage_ids_repetes = comptage_ids[comptage_ids > 1]

display(comptage_ids_repetes.head(10))

id_pdc_itinerance
Non concerné       118
FRLIBE3126085        4
FRCG0E002113         3
FRCG0E002114         3
FRCG0E002115         3
FRCG0E002116         3
FRCG0E001353         3
FRALLEGO9004771      3
FRALLEGO9004772      3
FRALLEGO9004773      3
Name: count, dtype: int64

In [18]:
# Garder uniquement les vrais identifiants répétés
comptage_ids_repetes_valides = comptage_ids[
    (comptage_ids > 1)
    & (comptage_ids.index != "Non concerné")
]

display(comptage_ids_repetes_valides.head(10))

id_pdc_itinerance
FRLIBE3126085      4
FRCG0E002113       3
FRCG0E002114       3
FRCG0E002115       3
FRCG0E002116       3
FRCG0E001353       3
FRALLEGO9004771    3
FRALLEGO9004772    3
FRALLEGO9004773    3
FRCG0E001355       3
Name: count, dtype: int64

In [19]:
id_exemple = comptage_ids_repetes_valides.index[0]

colonnes_analyse = [
    "id_pdc_itinerance",
    "id_station_itinerance",
    "nom_station",
    "adresse_station",
    "puissance_nominale",
    "date_maj",
    "last_modified",
    "datagouv_resource_id",
    "datagouv_organization_or_owner",
    "consolidated_longitude",
    "consolidated_latitude"
]

display(
    df.loc[
        df["id_pdc_itinerance"] == id_exemple,
        colonnes_analyse
    ].sort_values("date_maj")
)

,id_pdc_itinerance,id_station_itinerance,nom_station,adresse_station,puissance_nominale,date_maj,last_modified,datagouv_resource_id,datagouv_organization_or_owner,consolidated_longitude,consolidated_latitude
105427,FRLIBE3126085,FRLIBE3126085,Residence Carouge,34 Rue de lâÃÃ´Orge 91220 Brtigny-sur-Orge,22.0,2023-10-11,2026-01-09T13:01:51.509000+00:00,7514352c-c1ad-4530-8dc5-bf05bac9a786,eoliberty,2.297193,48.618492
105428,FRLIBE3126085,FRLIBE3126085,Residence Carouge,34 Rue de l‚Äö√Ñ√¥Orge 91220 Brétigny-sur-Orge,22.0,2023-10-11,2025-11-24T21:47:43.507000+00:00,04b89d29-2904-42c9-9e6b-f3c7dcd988ec,greenea,2.297193,48.618492
105425,FRLIBE3126085,FRLIBE3126085,Résidence Carouge,34 rue de l'Orge,7.4,2023-10-21,2024-12-06T06:47:29.008000+00:00,2f2c05b7-3ea6-461b-a6ab-7d5a3fed0718,bornevo,2.300000,48.620000
105426,FRLIBE3126085,FRLIBE3126085,Residence Carouge,34 Rue de l’Orge 91220 Brétigny-sur-Orge,22.0,2023-11-10,2024-12-06T06:47:51.017000+00:00,26160785-4f6d-428e-bcbb-9172278919ed,charge-in,2.297193,48.618492


In [20]:
# Sous-ensemble des lignes avec un véritable identifiant répété
ids_repetes_valides = comptage_ids_repetes_valides.index

df_repetes = df[
    df["id_pdc_itinerance"].isin(ids_repetes_valides)
].copy()

# Analyse globale par identifiant
analyse_groupes = df_repetes.groupby("id_pdc_itinerance").agg(
    nombre_lignes=("id_pdc_itinerance", "size"),
    nombre_sources=("datagouv_resource_id", "nunique"),
    nombre_organisations=("datagouv_organization_or_owner", "nunique"),
    nombre_puissances=("puissance_nominale", "nunique"),
    nombre_longitudes=("consolidated_longitude", "nunique"),
    nombre_latitudes=("consolidated_latitude", "nunique"),
    nombre_dates_maj=("date_maj", "nunique")
)

display(analyse_groupes.head())

,nombre_lignes,nombre_sources,nombre_organisations,nombre_puissances,nombre_longitudes,nombre_latitudes,nombre_dates_maj
id_pdc_itinerance,,,,,,,
ATHTBE1012062,2,2,2,2,2,2,2
ATHTBE1012063,2,2,2,2,2,2,2
FR190E190001SPE24ABB360121,2,2,2,1,2,2,2
FR190E190001SPE24ABB360122,2,2,2,1,2,2,2
FR190E190001SPE24ABB360341,2,2,2,1,2,2,2


In [21]:
resume_doublons = {
    "Groupes répétés analysés": len(analyse_groupes),
    "Groupes avec plusieurs sources": (analyse_groupes["nombre_sources"] > 1).sum(),
    "Groupes avec plusieurs organisations": (analyse_groupes["nombre_organisations"] > 1).sum(),
    "Groupes avec plusieurs puissances": (analyse_groupes["nombre_puissances"] > 1).sum(),
    "Groupes avec plusieurs longitudes": (analyse_groupes["nombre_longitudes"] > 1).sum(),
    "Groupes avec plusieurs latitudes": (analyse_groupes["nombre_latitudes"] > 1).sum(),
    "Groupes avec plusieurs dates de mise à jour": (analyse_groupes["nombre_dates_maj"] > 1).sum()
}

for cle, valeur in resume_doublons.items():
    print(f"{cle} : {valeur}")

Groupes répétés analysés : 63869
Groupes avec plusieurs sources : 63869
Groupes avec plusieurs organisations : 63869
Groupes avec plusieurs puissances : 9719
Groupes avec plusieurs longitudes : 32861
Groupes avec plusieurs latitudes : 33643
Groupes avec plusieurs dates de mise à jour : 59716


### Conclusion provisoire sur les identifiants répétés

L'analyse montre que les répétitions peuvent correspondre à :

- plusieurs publications provenant de sources différentes ;
- plusieurs versions avec des dates de mise à jour différentes ;
- des conflits sur la puissance ;
- des écarts de coordonnées ;
- des organisations productrices différentes.

Ces répétitions ne doivent donc pas être supprimées automatiquement. La règle de dédoublonnage sera définie à partir de ces résultats globaux.


## 2. Définition de la règle de dédoublonnage

L'analyse montre que les identifiants répétés proviennent systématiquement de plusieurs ressources et organisations.

La règle retenue est la suivante :

- conserver les lignes dont l'identifiant n'est pas répété ;
- pour les identifiants répétés sans conflit majeur, conserver la version ayant la `date_maj` la plus récente ;
- en cas d'égalité sur `date_maj`, utiliser `last_modified` comme second critère ;
- isoler les groupes présentant plusieurs puissances ou des coordonnées différentes afin de conserver une trace des conflits.

Cette règle évite un `drop_duplicates()` aveugle.

Quel est le problème ?

Le fichier contient plusieurs lignes avec le même id_pdc_itinerance
Mais ces lignes ne sont pas identiques.

=>plusieurs sources publient des informations sur leMEME point de charge.

pas des doublons stricts. mais plusieurs versions ou publications duMEME objet

Pourquoi faut-il dédoublonner ?
1 point de charge = 1 ligne
choisir quelle version conserver

La règle la plus simple

La règle la plus facile à comprendre et à défendre est :

Pour chaque id_pdc_itinerance, conserver la ligne ayant la date_maj la plus récente

Pourquoi utiliser last_modified ?

Parfois, deux lignes du même identifiant peuvent avoir exactement la même date_maj

Pourquoi conserver une table de conflits ?

Certains identifiants répétés contiennent des informations contradictoires : plusieurs puissances oordonnées et adresses

garde les lignes contradictoires dans une table séparée pour la tracabilité 
ATTENTION LA LIGNE RETENU PEUT AVOIR DES ERREURS !!!!


Resumer
Même identifiant répété =>Comparer les dates=> garder la date_maj la plus récente
      
Si égalité : garder le last_modified le plus récent
        
PS : Conserver les cas contradictoires dans une table de conflits

La décision métier centrale est donc :

Une ligne finale représente la version la plus récente connue d’un point de charge.

# Sauvegarder la version brute

In [22]:
# Sauvegarde du dataset avant dédoublonnage
df_brut = df.copy()

print("Nombre de lignes avant traitement :", len(df_brut))

Nombre de lignes avant traitement : 227232


# Convertir les dates

In [23]:
# Conversion des dates utilisées pour choisir la version la plus récente

df["date_maj"] = pd.to_datetime(
    df["date_maj"],
    errors="coerce",
    utc=True
)

df["last_modified"] = pd.to_datetime(
    df["last_modified"],
    errors="coerce",
    utc=True
)

pour pouvoir les trier correctementmmmmmmmm

# Mettre de côté les identifiants conflictuels

In [24]:
# Identifiants présentant des informations contradictoires

ids_conflit = analyse_groupes.loc[
    (analyse_groupes["nombre_puissances"] > 1)
    | (analyse_groupes["nombre_longitudes"] > 1)
    | (analyse_groupes["nombre_latitudes"] > 1)
].index

df_conflits = df[
    df["id_pdc_itinerance"].isin(ids_conflit)
].copy()

print("Nombre d'identifiants conflictuels :", len(ids_conflit))
print("Nombre de lignes conservées dans df_conflits :", len(df_conflits))

Nombre d'identifiants conflictuels : 37115
Nombre de lignes conservées dans df_conflits : 77246


77 246 lignes dans df_conflits = toutes les lignes correspondant à ces 37 115 bornes.(meme bornes on peut avoir deux sources differebtes

# Séparer les identifiants exploitables des identifiants non exploitables


In [25]:
# Séparer les identifiants exploitables des identifiants non exploitables

mask_id_valide = (
    df["id_pdc_itinerance"].notna()
    & (df["id_pdc_itinerance"].str.strip() != "")
    & (df["id_pdc_itinerance"] != "Non concerné")
)

df_ids_valides = df[mask_id_valide].copy()
df_ids_non_valides = df[~mask_id_valide].copy()

print("Lignes avec identifiant valide :", len(df_ids_valides))
print("Lignes avec identifiant non exploitable :", len(df_ids_non_valides))

Lignes avec identifiant valide : 227114
Lignes avec identifiant non exploitable : 118


La logique est donc :

dédoublonner uniquement df_ids_valides ;
garder df_ids_non_valides à part ;
les réintégrer ensuite sans les dédoublonner.

# 1. Trie par versions

In [26]:
df_ids_valides = df_ids_valides.sort_values(
    by=[
        "id_pdc_itinerance",
        "date_maj",
        "last_modified"
    ],
    ascending=True,
    na_position="first"
)
#Garder la version la plus récente
df_ids_dedoublonnes = df_ids_valides.drop_duplicates(
    subset=["id_pdc_itinerance"],
    keep="last"
)
print("Lignes avec identifiants valides avant :", len(df_ids_valides))
print("Lignes avec identifiants valides après :", len(df_ids_dedoublonnes))
print(
    "Lignes retirées :",
    len(df_ids_valides) - len(df_ids_dedoublonnes)
)

Lignes avec identifiants valides avant : 227114
Lignes avec identifiants valides après : 160137
Lignes retirées : 66977


In [27]:
# on reconstitut le dataset 
df = pd.concat(
    [
        df_ids_dedoublonnes,
        df_ids_non_valides
    ],
    ignore_index=True
)

print("Lignes avant :", len(df_brut))
print("Lignes après :", len(df))
print("Lignes retirées :", len(df_brut) - len(df))

Lignes avant : 227232
Lignes après : 160255
Lignes retirées : 66977


In [28]:
mask_id_valide_final = (
    df["id_pdc_itinerance"].notna()
    & (df["id_pdc_itinerance"].str.strip() != "")
    & (df["id_pdc_itinerance"] != "Non concerné")
)

repetitions_restantes = (
    df.loc[
        mask_id_valide_final,
        "id_pdc_itinerance"
    ]
    .duplicated()
    .sum()
)

print("Identifiants valides répétés restants :", repetitions_restantes)

Identifiants valides répétés restants : 0


### Résultat du dédoublonnage

Le dataset contenait initialement **227 232 lignes**.

Après application de la règle de dédoublonnage :

- **160 137 lignes** avec un identifiant valide ont été conservées ;
- **118 lignes** avec un identifiant non exploitable ont été conservées sans dédoublonnage ;
- le dataset final contient **160 255 lignes** ;
- **66 977 lignes** ont été retirées.

La règle appliquée consiste à conserver, pour chaque `id_pdc_itinerance` valide, la version la plus récente selon `date_maj`, puis `last_modified` en cas d’égalité.

Les lignes dont l’identifiant est vide ou égal à `Non concerné` ont été conservées afin d’éviter de supprimer des bornes potentiellement différentes.

In [29]:
# Vérifier les identifiants non exploitables conservés dans df

mask_non_valides_final = (
    df["id_pdc_itinerance"].isna()
    | (df["id_pdc_itinerance"].astype("string").str.strip() == "")
    | (df["id_pdc_itinerance"] == "Non concerné")
)

print(
    "Nombre de lignes non exploitables conservées dans df :",
    mask_non_valides_final.sum()
)

display(
    df.loc[
        mask_non_valides_final,
        ["id_pdc_itinerance", "nom_station", "adresse_station"]
    ].head(20)
)

Nombre de lignes non exploitables conservées dans df : 118


,id_pdc_itinerance,nom_station,adresse_station
160137,Non concerné,Aubade - Comptoir des Fers,"1 Rue des Biches, 74100 Ville-la-Grand"
160138,Non concerné,Parking MAMBHOME,1 Impasse de Recouvrance 17100 SAINTES
160139,Non concerné,"BORNE SECURITEST, DOUVRES LA DELIVRANDE",Rue Jean Perrin 14440 Douvres-la-Délivrande
160140,Non concerné,La jabotte,13 Avenue Max Maurey 06160 Antibes
160141,Non concerné,NANTEUIL,12 Av. Fridingen 77100 Nanteuil-lès-Meaux France
160142,Non concerné,Parking P4 électrique,"2 Rue de Finlande, 69124 Colombier-Saugnieu"
160143,Non concerné,Parking Boulangerie,6 rue edouard branly
160144,Non concerné,Absolut Bougival,21 chemin du haut du trou martin
160145,Non concerné,ALC CAR,2 RUE BRANLY 77400
160146,Non concerné,"Borne, Domaine, Charton-Vachet",25 rue du May


## Conclusion de la partie 2 — Grain et doublons

Le fichier contient plusieurs lignes pour un même `id_pdc_itinerance`, car les données proviennent de plusieurs ressources et livraisons.

La règle retenue consiste à :

- conserver une copie du dataset brut ;
- exclure temporairement les identifiants vides ou égaux à `Non concerné` du dédoublonnage ;
- trier les versions selon `date_maj`, puis `last_modified` ;
- conserver la version la plus récente pour chaque identifiant valide ;
- conserver séparément les groupes présentant des conflits ;
- réintégrer les lignes sans identifiant exploitable.

Après dédoublonnage, le dataset contient 160 255 lignes et aucun identifiant valide ne reste répété.

# Partie 3 — Nettoyage et enrichissement